In [5]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sqlalchemy.engine import URL

In [17]:
# 1. Connect to MySQL Database
connection_url = URL.create(
    "mysql+pymysql",
    username="root",
    password="stanly@1234",
    host="localhost",
    port=3306,
    database="ecommerce_db"
)

engine = create_engine(connection_url)

# 2. Extract Customer Data from your SQL View
query = "SELECT * FROM view_customer_rfm_segments"
df = pd.read_sql(query, con=engine)

# 3. Define Binary Churn Target (e.g., Recency > 90 days = Churned)
# Demonstrates setting actionable business thresholds
df['is_churned'] = (df['recency_days'] > 90).astype(int)

# Feature Matrix & Target Vector
X = df[['recency_days', 'frequency_count', 'net_revenue']]
y = df['is_churned']

# 4. Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Fit Interpretable Logistic Regression
# Prioritizing interpretability over complex black-box models (matches JPMC requirement)
model = LogisticRegression()
model.fit(X_train, y_train)

# 6. Evaluate Model Performance
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("--- Model Performance Metrics ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

# 7. Output Feature Importance / Odds Ratios
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0],
    'Odds_Ratio': np.exp(model.coef_[0])
})
print("\n--- Feature Importance (Odds Ratios) ---")
print(coefficients)

--- Model Performance Metrics ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       582
           1       1.00      1.00      1.00       275

    accuracy                           1.00       857
   macro avg       1.00      1.00      1.00       857
weighted avg       1.00      1.00      1.00       857

ROC-AUC Score: 1.0000

--- Feature Importance (Odds Ratios) ---
           Feature  Coefficient  Odds_Ratio
0     recency_days     2.383607   10.843951
1  frequency_count     0.249983    1.284004
2      net_revenue     0.000149    1.000149


In [19]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix

# 1. Connect to MySQL Database
engine = create_engine(connection_url)


# 2. Dynamic SQL Query with 90-Day Cutoff relative to Max Date
query = """
WITH max_date_cte AS (
    SELECT MAX(transaction_date) AS max_date FROM fact_transactions
),
cutoff_date_cte AS (
    SELECT DATE_SUB(max_date, INTERVAL 90 DAY) AS cutoff_date FROM max_date_cte
),
observation_period AS (
    -- Calculate features using data strictly BEFORE the cutoff date
    SELECT 
        f.customer_id,
        DATEDIFF((SELECT cutoff_date FROM cutoff_date_cte), MAX(f.transaction_date)) AS recency_days,
        COUNT(DISTINCT f.invoice_id) AS frequency_count,
        SUM(f.line_total) AS monetary_value
    FROM fact_transactions f, cutoff_date_cte c
    WHERE f.transaction_date < c.cutoff_date
    GROUP BY f.customer_id
    HAVING SUM(f.line_total) > 0.01
),
performance_period AS (
    -- Check for purchases AFTER the cutoff date
    SELECT DISTINCT f.customer_id
    FROM fact_transactions f, cutoff_date_cte c
    WHERE f.transaction_date >= c.cutoff_date
)
SELECT 
    o.customer_id,
    o.recency_days,
    o.frequency_count,
    ROUND(o.monetary_value, 2) AS net_revenue,
    CASE WHEN p.customer_id IS NULL THEN 1 ELSE 0 END AS is_churned
FROM observation_period o
LEFT JOIN performance_period p ON o.customer_id = p.customer_id;
"""

df = pd.read_sql(query, con=engine)

# Quick check to ensure we have both classes before modeling
print("Class Distribution in Dataset:")
print(df['is_churned'].value_counts())

# 3. Define Features (X) & Target (y)
X = df[['recency_days', 'frequency_count', 'net_revenue']]
y = df['is_churned']

# 4. Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Build Pipeline (StandardScaler + Logistic Regression)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
])

# 6. Fit Model
pipeline.fit(X_train, y_train)

# 7. Model Evaluation
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC Score (Average Precision): {average_precision_score(y_test, y_prob):.4f}")

# 8. Feature Importance (Odds Ratios)
coefs = pipeline.named_steps['classifier'].coef_[0]
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': coefs,
    'Odds_Ratio': np.exp(coefs)
}).sort_values(by='Odds_Ratio', ascending=False)

print("\n=== Feature Importance (Odds Ratios) ===")
print(importance_df)

Class Distribution in Dataset:
is_churned
0    1969
1    1384
Name: count, dtype: int64

=== Confusion Matrix ===
[[225 169]
 [ 75 202]]

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.75      0.57      0.65       394
           1       0.54      0.73      0.62       277

    accuracy                           0.64       671
   macro avg       0.65      0.65      0.64       671
weighted avg       0.67      0.64      0.64       671

ROC-AUC Score: 0.6901
PR-AUC Score (Average Precision): 0.5400

=== Feature Importance (Odds Ratios) ===
           Feature  Coefficient  Odds_Ratio
0     recency_days     0.326062    1.385501
2      net_revenue    -0.412368    0.662081
1  frequency_count    -1.586586    0.204623


In [21]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix

# 1. Connect to MySQL Database
engine = create_engine(connection_url)

# 2. Dynamic SQL Query with Advanced Behavioral Feature Engineering
query = """
WITH max_date_cte AS (
    SELECT MAX(transaction_date) AS max_date FROM fact_transactions
),
cutoff_date_cte AS (
    SELECT DATE_SUB(max_date, INTERVAL 90 DAY) AS cutoff_date FROM max_date_cte
),
observation_period AS (
    SELECT 
        f.customer_id,
        DATEDIFF((SELECT cutoff_date FROM cutoff_date_cte), MAX(f.transaction_date)) AS recency_days,
        DATEDIFF((SELECT cutoff_date FROM cutoff_date_cte), MIN(f.transaction_date)) AS tenure_days,
        COUNT(DISTINCT f.invoice_id) AS frequency_count,
        SUM(f.line_total) AS net_revenue,
        AVG(f.line_total) AS avg_item_spend,
        COUNT(f.line_total) AS total_items_bought
    FROM fact_transactions f, cutoff_date_cte c
    WHERE f.transaction_date < c.cutoff_date
    GROUP BY f.customer_id
    HAVING SUM(f.line_total) > 0.01
),
performance_period AS (
    SELECT DISTINCT f.customer_id
    FROM fact_transactions f, cutoff_date_cte c
    WHERE f.transaction_date >= c.cutoff_date
)
SELECT 
    o.customer_id,
    o.recency_days,
    o.tenure_days,
    o.frequency_count,
    ROUND(o.net_revenue, 2) AS net_revenue,
    ROUND(o.net_revenue / o.frequency_count, 2) AS avg_order_value,
    ROUND(o.frequency_count / GREATEST(o.tenure_days, 1), 4) AS purchase_velocity,
    ROUND(o.recency_days / GREATEST(o.tenure_days, 1), 4) AS recency_tenure_ratio,
    CASE WHEN p.customer_id IS NULL THEN 1 ELSE 0 END AS is_churned
FROM observation_period o
LEFT JOIN performance_period p ON o.customer_id = p.customer_id;
"""

df = pd.read_sql(query, con=engine)

# 3. Features & Target
feature_cols = [
    'recency_days', 'tenure_days', 'frequency_count', 
    'net_revenue', 'avg_order_value', 'purchase_velocity', 'recency_tenure_ratio'
]
X = df[feature_cols]
y = df['is_churned']

# 4. Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Model 1: Scaled Logistic Regression (Interpretable Baseline)
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
])
lr_pipeline.fit(X_train, y_train)

# 6. Model 2: Random Forest Classifier (Non-linear Benchmark)
rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=5, 
    class_weight='balanced', 
    random_state=42
)
rf_model.fit(X_train, y_train)

# 7. Evaluate Logistic Regression
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

print("==========================================")
print("  LOGISTIC REGRESSION PERFORMANCE Metrics ")
print("==========================================")
print(classification_report(y_test, y_pred_lr))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_lr):.4f}")
print(f"PR-AUC Score : {average_precision_score(y_test, y_prob_lr):.4f}")

# 8. Evaluate Random Forest
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("\n==========================================")
print("  RANDOM FOREST PERFORMANCE Metrics       ")
print("==========================================")
print(classification_report(y_test, y_pred_rf))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_rf):.4f}")
print(f"PR-AUC Score : {average_precision_score(y_test, y_prob_rf):.4f}")

# 9. Extract Logistic Regression Feature Importances (Odds Ratios)
coefs = lr_pipeline.named_steps['classifier'].coef_[0]
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': coefs,
    'Odds_Ratio': np.exp(coefs)
}).sort_values(by='Odds_Ratio', ascending=False)

print("\n=== Logistic Regression Feature Importance (Odds Ratios) ===")
print(importance_df.to_string(index=False))

  LOGISTIC REGRESSION PERFORMANCE Metrics 
              precision    recall  f1-score   support

           0       0.74      0.61      0.67       394
           1       0.56      0.70      0.62       277

    accuracy                           0.65       671
   macro avg       0.65      0.65      0.64       671
weighted avg       0.67      0.65      0.65       671

ROC-AUC Score: 0.6996
PR-AUC Score : 0.5519

  RANDOM FOREST PERFORMANCE Metrics       
              precision    recall  f1-score   support

           0       0.73      0.63      0.68       394
           1       0.56      0.67      0.61       277

    accuracy                           0.65       671
   macro avg       0.65      0.65      0.65       671
weighted avg       0.66      0.65      0.65       671

ROC-AUC Score: 0.7072
PR-AUC Score : 0.5718

=== Logistic Regression Feature Importance (Odds Ratios) ===
             Feature  Coefficient  Odds_Ratio
        recency_days     0.347885    1.416070
recency_tenure_ra